# **Autonomous Data Pipeline Incident Agent (POC)**

**Overview:** This proof of concept demonstrates an AI agent that acts as the "first responder" for data pipeline incidents. The agent:

1. **Investigates:** Checks the pipeline's health (last run status, data freshness/lag, failed record count) and fetches the recent job logs when a user reports an issue.
2. **Acts:** If the job failed or stalled due to a transient issue (e.g. a timeout or a stuck task) or data lag is critical (>90 minutes behind), it **Reruns** the pipeline job.
3. **Escalates:** If the logs show a problem a rerun won't fix (e.g. a schema mismatch, a data quality check failure, or an unreachable upstream source), it **Escalates** to a human data engineer.


In [1]:
import os
import json
from openai import OpenAI
from google.colab import userdata

In [3]:
# Initialize OpenAI Client (pointed at OpenRouter)
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get('OPENROUTER_APIKEY'),
)

In [4]:
# --- Check Pipeline Health ---
def get_pipeline_health(pipeline_id: str) -> str:
    """Returns the last run status, data lag and failed record count for a given pipeline."""
    print(f"-> TOOL: Checking health for {pipeline_id}...")

    metrics = {
        # Scenario 1: Job stalled on a transient timeout, data is stale (Needs Rerun)
        "sales-etl-daily": {"last_run_status": "Failed", "data_lag_minutes": 185, "failed_records": 0, "status": "Warning"},

        # Scenario 2: Healthy (No Action Needed)
        "customer-sync-02": {"last_run_status": "Success", "data_lag_minutes": 5, "failed_records": 0, "status": "Healthy"},

        # Scenario 3: Schema mismatch from source (Needs Escalation)
        "orders-ingest-03": {"last_run_status": "Failed", "data_lag_minutes": 60, "failed_records": 12450, "status": "Critical"},

        # Scenario 4: Upstream source unreachable (Needs Escalation)
        "clickstream-stream-09": {"last_run_status": "Failed", "data_lag_minutes": 240, "failed_records": 0, "status": "Error"},

        # Scenario 5: Completely Normal
        "inventory-agg-04": {"last_run_status": "Success", "data_lag_minutes": 10, "failed_records": 0, "status": "Healthy"},
    }

    result = metrics.get(pipeline_id, {"error": "Pipeline not found. Check the ID."})
    return json.dumps(result)


In [5]:
def fetch_recent_logs(pipeline_id: str, lines: int = 5) -> str:
    """Returns the last N lines of job logs."""
    print(f"-> TOOL: Fetching last {lines} log lines for {pipeline_id}...")

    # Different logs for different pipelines to trigger different agent behaviors
    log_database = {
        "sales-etl-daily": [
            "[INFO] Job run #4521 started",
            "[INFO] Extract step completed: 1,204,332 rows",
            "[WARN] Transform task 'aggregate_sales' running longer than expected",
            "[ERROR] Task 'aggregate_sales' timed out after 3600s (worker lost heartbeat)",
            "[CRITICAL] Job stalled, downstream tables not refreshed"
        ],
        "customer-sync-02": [
            "[INFO] Job run #882 started",
            "[INFO] Extracted 15,220 rows from CRM",
            "[INFO] Data quality checks passed (12/12)",
            "[INFO] Loaded 15,220 rows into warehouse",
            "[INFO] Job run #882 completed successfully"
        ],
        "orders-ingest-03": [
            "[INFO] Job run #3310 started",
            "[WARN] Unexpected column 'order_total_usd' in source file",
            "[ERROR] SchemaMismatchError: expected column 'order_total' (DECIMAL) not found",
            "[ERROR] Data quality check failed: 12,450 rows rejected",
            "[CRITICAL] Load aborted to protect target table"
        ],
        "clickstream-stream-09": [
            "[INFO] Consumer group started",
            "[ERROR] Connection refused: kafka-broker-main:9092",
            "[ERROR] Failed to fetch offsets for topic 'clickstream-events'",
            "[CRITICAL] Upstream source unreachable: event stream is down",
            "[ERROR] Retrying in 30s..."
        ],
        "inventory-agg-04": [
            "[INFO] Job run #77 started",
            "[INFO] Read 48,900 rows from inventory_raw",
            "[INFO] Data quality checks passed (8/8)",
            "[INFO] Wrote aggregates to inventory_daily",
            "[INFO] Job run #77 completed successfully"
        ]
    }

    # Default logs if pipeline not found in specific list
    default_logs = ["[INFO] Pipeline idle", "[INFO] Scheduler heartbeat received"]

    logs = log_database.get(pipeline_id, default_logs)
    return json.dumps({"logs": logs[:lines]})

In [6]:
def rerun_pipeline(pipeline_id: str) -> str:
    """Triggers a rerun of the given pipeline job and returns a confirmation."""
    print("-> TOOL: Rerunning pipeline job...")
    return json.dumps({"status": "success", "message": "Pipeline rerun triggered successfully"})


In [7]:
def escalate_to_engineer(summary: str) -> str:
    """Creates an escalation ticket for a human data engineer with the incident summary."""
    print("-> TOOL: Escalating to data engineer...")
    return json.dumps({"status": "success", "message": "Ticket created"})

In [8]:
AVAILABLE_FUNCTIONS = {
    "get_pipeline_health": get_pipeline_health,
    "fetch_recent_logs": fetch_recent_logs,
    "rerun_pipeline": rerun_pipeline,
    "escalate_to_engineer": escalate_to_engineer,
}

In [9]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_pipeline_health",
            "description": "Checks the last run status, data freshness lag (minutes) and failed record count of a specific data pipeline.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pipeline_id": {"type": "string", "description": "The ID of the pipeline, e.g., 'sales-etl-daily'"}
                },
                "required": ["pipeline_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_recent_logs",
            "description": "Retrieves the most recent job log entries from a pipeline to diagnose errors.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pipeline_id": {"type": "string", "description": "The ID of the pipeline."},
                    "lines": {"type": "integer", "description": "Number of log lines to fetch."}
                },
                "required": ["pipeline_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "rerun_pipeline",
            "description": "Reruns the pipeline job.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pipeline_id": {"type": "string", "description": "The ID of the pipeline, e.g., 'sales-etl-daily'"}
                },
                "required": ["pipeline_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "escalate_to_engineer",
            "description": "Escalates the issue to a human data engineer when a rerun won't fix it or the error is unknown.",
            "parameters": {
                "type": "object",
                "properties": {
                     "summary" : {'type': "string", "description":"The summary of the incident"}
                },
                "required": ["summary"]
            }
        }
    }
]

In [10]:
def run_it_agent(user_issue: str):
    print(f"\n--- New Incident: {user_issue} ---")

    messages = [
        {"role": "system", "content": "You are a Level 1 Data Pipeline Responder. Investigate data pipeline issues. "
                                      "If a job failed or stalled due to a transient issue (like a timeout or stuck task) or data lag is > 90 minutes, rerun the pipeline. "
                                      "If logs show a problem a rerun won't fix (like a schema mismatch, a data quality check failure, or an unreachable upstream source), escalate to a data engineer."},
        {"role": "user", "content": user_issue}
    ]

    while True:
        print("\n[AI Thinking...]")
        response = client.chat.completions.create(
            model="openai/gpt-6-luna",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        response_msg = response.choices[0].message
        messages.append(response_msg)

        if response_msg.tool_calls:
            for tool_call in response_msg.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # Retrieve the actual python function based on name
                function_to_call = AVAILABLE_FUNCTIONS.get(func_name)

                if function_to_call:
                    # Execute the function
                    tool_output = function_to_call(**func_args)

                    messages.append({
                         "role": "tool",
                         "tool_call_id": tool_call.id,
                         "name": func_name,
                         "content": tool_output
                    })

        else:
            print(f"\n[FINAL RESPONSE]: {response_msg.content}")
            break


## TEST SCENARIOS

In [11]:
# Scenario A: Should trigger a rerun (job timed out, data is 185 min stale)
run_it_agent("The sales dashboard hasn't refreshed since this morning. Can you check sales-etl-daily?")


--- New Incident: The sales dashboard hasn't refreshed since this morning. Can you check sales-etl-daily? ---

[AI Thinking...]
-> TOOL: Checking health for sales-etl-daily...
-> TOOL: Fetching last 20 log lines for sales-etl-daily...

[AI Thinking...]
-> TOOL: Rerunning pipeline job...

[AI Thinking...]

[FINAL RESPONSE]: `sales-etl-daily` failed because the `aggregate_sales` task timed out after losing its worker heartbeat, and the dashboard data was 185 minutes behind. I triggered a rerun successfully.


In [12]:
# Scenario B: Healthy pipeline (agent should investigate and report no action needed)
run_it_agent("Something seems off with customer-sync-02")


--- New Incident: Something seems off with customer-sync-02 ---

[AI Thinking...]
-> TOOL: Checking health for customer-sync-02...
-> TOOL: Fetching last 20 log lines for customer-sync-02...

[AI Thinking...]

[FINAL RESPONSE]: `customer-sync-02` appears healthy: its last run succeeded, data lag is 5 minutes, and there are no failed records. The latest logs show all 15,220 rows loaded and all data quality checks passed. No rerun or escalation is needed.


In [13]:
# Scenario C: The Schema Mismatch Case (orders-ingest-03)
# Agent should see failed records + SchemaMismatchError logs -> Escalate (a rerun won't fix it)
run_it_agent("The orders table is missing today's data. Check orders-ingest-03.")

print("\n" + "="*50 + "\n")


--- New Incident: The orders table is missing today's data. Check orders-ingest-03. ---

[AI Thinking...]
-> TOOL: Checking health for orders-ingest-03...

[AI Thinking...]
-> TOOL: Fetching last 10 log lines for orders-ingest-03...

[AI Thinking...]
-> TOOL: Escalating to data engineer...

[AI Thinking...]

[FINAL RESPONSE]: orders-ingest-03 failed. The source has `order_total_usd` instead of the expected `order_total` (DECIMAL), and 12,450 rows failed data-quality checks. The load was aborted, so a rerun is unlikely to help until the schema issue is fixed. I escalated this to a data engineer.




In [14]:
# Scenario D: The Upstream Source Failure (clickstream-stream-09)
# Agent should see "Connection refused" / source unreachable logs -> Escalate
run_it_agent("Clickstream analytics aren't updating. Can you check clickstream-stream-09?")

print("\n" + "="*50 + "\n")


--- New Incident: Clickstream analytics aren't updating. Can you check clickstream-stream-09? ---

[AI Thinking...]
-> TOOL: Checking health for clickstream-stream-09...

[AI Thinking...]
-> TOOL: Fetching last 10 log lines for clickstream-stream-09...

[AI Thinking...]
-> TOOL: Escalating to data engineer...

[AI Thinking...]

[FINAL RESPONSE]: clickstream-stream-09 is failed and 240 minutes behind. The logs show the Kafka broker connection is refused and the upstream event stream is unreachable, so a rerun is unlikely to help. I escalated the issue to a data engineer.




In [15]:
# Scenario E: The Healthy Pipeline (inventory-agg-04)
# Agent should see a successful run and low lag -> Do nothing / Report healthy
run_it_agent("Check inventory-agg-04 just to be safe.")


--- New Incident: Check inventory-agg-04 just to be safe. ---

[AI Thinking...]
-> TOOL: Checking health for inventory-agg-04...

[AI Thinking...]

[FINAL RESPONSE]: `inventory-agg-04` is healthy: the last run succeeded, data lag is 10 minutes, and there are no failed records. No action needed.
